**Импорты**

In [10]:
import os, glob, zipfile
import numpy as np
from collections import defaultdict

from datasets import Dataset, DatasetDict, concatenate_datasets
from sklearn.model_selection import train_test_split

from transformers import (
    AutoTokenizer,
    AutoModelForTokenClassification,
    TrainingArguments,
    Trainer,
    EarlyStoppingCallback,
)
from seqeval.metrics import f1_score, precision_score, recall_score

**Константы**

In [24]:
ARCHIVE_PATH = "ru-ACTER.zip"
EXTRACT_DIR  = "data"
ROOT_DIR     = "data/ru-ACTER/ru"

MODEL_NAME   = "ai-forever/ruBert-base"
MAX_LENGTH = 128
BATCH_SIZE   = 8
LEARNING_RATE = 2e-5
EPOCHS       = 10
SEED         = 42
VAL_SIZE     = 0.1
FP16 = True

DOMAIN_MAP = {
    "corp": "corruption",
    "equi": "equestrian",
    "htfl": "heart_failure",
    "wind": "wind_energy",
}

label2id = {"O": 0, "B": 1, "I": 2}
id2label = {v: k for k, v in label2id.items()}

**Разархивирование датасета**

In [12]:
if not os.path.exists(os.path.join(EXTRACT_DIR, "ru-ACTER", "ru")):
    with zipfile.ZipFile(ARCHIVE_PATH, "r") as zf:
        zf.extractall(EXTRACT_DIR)

**Загрузка всех BIO‑файлов**

In [13]:
def read_bio_file(filepath):
    sentences, labels = [], []
    with open(filepath, "r", encoding="utf-8") as f:
        cur_tokens, cur_labels = [], []
        for line in f:
            line = line.strip()
            if not line:
                if cur_tokens:
                    sentences.append(cur_tokens)
                    labels.append(cur_labels)
                    cur_tokens, cur_labels = [], []
                continue
            parts = line.split("\t")
            if len(parts) != 2:
                continue
            token, label = parts
            cur_tokens.append(token)
            cur_labels.append(label)
        if cur_tokens:
            sentences.append(cur_tokens)
            labels.append(cur_labels)
    return sentences, labels

bio_files = glob.glob(os.path.join(ROOT_DIR, "**/iob_annotations/with_named_entities/*.txt"), recursive=True)

domain_data = defaultdict(list)
for fpath in bio_files:
    domain = None
    for part in fpath.split(os.sep):
        if part in DOMAIN_MAP:
            domain = DOMAIN_MAP[part]
            break
    if domain is None:
        continue
    sents, labs = read_bio_file(fpath)
    for tokens, labels in zip(sents, labs):
        domain_data[domain].append({"tokens": tokens, "labels": labels})

datasets = {dom: Dataset.from_list(examples) for dom, examples in domain_data.items()}

# Пример
first_dom = list(datasets.keys())[0]
print(f"Домен: {first_dom}, предложений: {len(datasets[first_dom])}")
sample = datasets[first_dom][0]
print("Токены:", sample["tokens"][:10])
print("Метки :", sample["labels"][:10])

Домен: heart_failure, предложений: 2433
Токены: ['Митральный', 'стеноз', 'устранен', 'медикаментозным', 'лечением', 'сердечной', 'недостаточности', '.']
Метки : ['B', 'I', 'O', 'O', 'O', 'B', 'I', 'O']


**Статистический baseline на основе частоты**

In [29]:
from collections import Counter
from seqeval.metrics import f1_score, precision_score, recall_score

MAX_TERMS = 2000          # сколько самых частых n-грамм используем
N_MIN, N_MAX = 1, 5       # диапазон длин терминов

def extract_ngrams_from_tokens(tokens_list, n_min=N_MIN, n_max=N_MAX):
    """Извлекает все n-граммы, которые содержат хотя бы одну букву."""
    ngrams = []
    L = len(tokens_list)
    for start in range(L):
        for n in range(n_min, min(n_max + 1, L - start + 1)):
            ngram = tokens_list[start:start+n]

            if any(any(c.isalpha() for c in t) for t in ngram):
                ngrams.append(tuple(ngram))
    return ngrams


try:
    _ = domain_data
except NameError:
    domain_data = {
        dom: [{"tokens": ds[i]["tokens"], "labels": ds[i]["labels"]} for i in range(len(ds))]
        for dom, ds in datasets.items()
    }

results_freq = {}

for test_domain in domain_data.keys():
    train_domains = [d for d in domain_data if d != test_domain]

    train_samples = []
    for dom in train_domains:
        train_samples.extend(domain_data[dom])


    ngram_counter = Counter()
    for sample in train_samples:
        ngrams = extract_ngrams_from_tokens(sample["tokens"])
        ngram_counter.update(ngrams)

    if not ngram_counter:
        print(f"{test_domain}: нет n‑грамм в обучении, пропускаем")
        continue


    top_terms = set([term for term, _ in ngram_counter.most_common(MAX_TERMS)])


    y_true, y_pred = [], []
    for sample in domain_data[test_domain]:
        tokens = sample["tokens"]
        gold = sample["labels"]
        pred = ["O"] * len(tokens)
        for term in top_terms:
            tlen = len(term)
            for start in range(len(tokens) - tlen + 1):
                if tuple(tokens[start:start+tlen]) == term:
                    pred[start] = "B"
                    for k in range(1, tlen):
                        pred[start+k] = "I"
        y_true.append(gold)
        y_pred.append(pred)

    f1 = f1_score(y_true, y_pred)
    prec = precision_score(y_true, y_pred)
    rec = recall_score(y_true, y_pred)
    results_freq[test_domain] = {"f1": f1, "precision": prec, "recall": rec}
    print(f"{test_domain:20s}: F1={f1:.4f}, P={prec:.4f}, R={rec:.4f}")

# Средние метрики
avg_f1 = sum(r["f1"] for r in results_freq.values()) / len(results_freq)
avg_p = sum(r["precision"] for r in results_freq.values()) / len(results_freq)
avg_r = sum(r["recall"] for r in results_freq.values()) / len(results_freq)
print(f"{'Среднее':20s}: F1={avg_f1:.4f}, P={avg_p:.4f}, R={avg_r:.4f}")

heart_failure       : F1=0.0068, P=0.0048, R=0.0119
equestrian          : F1=0.0071, P=0.0048, R=0.0143
corruption          : F1=0.0057, P=0.0039, R=0.0107
wind_energy         : F1=0.0055, P=0.0035, R=0.0139
Среднее             : F1=0.0063, P=0.0042, R=0.0127


# Нейросеть

**Токенизатор и выравнивание меток**

In [14]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def tokenize_and_align_labels(examples, tokenizer, label2id, max_length=MAX_LENGTH):
    tokenized_inputs = tokenizer(
        examples["tokens"],
        is_split_into_words=True,
        truncation=True,
        padding="max_length",
        max_length=max_length,
        return_tensors=None,
    )
    labels_batch = []
    for i, label_seq in enumerate(examples["labels"]):
        word_ids = tokenized_inputs.word_ids(batch_index=i)
        aligned = []
        prev = None
        for wid in word_ids:
            if wid is None:
                aligned.append(-100)
            elif wid != prev:
                aligned.append(label2id.get(label_seq[wid], -100))
            else:
                aligned.append(-100)
            prev = wid
        labels_batch.append(aligned)
    tokenized_inputs["labels"] = labels_batch
    return tokenized_inputs

config.json:   0%|          | 0.00/590 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

**LODO-разбиения**

In [15]:
def create_lodo_splits(domain_data, test_domain, val_size=VAL_SIZE):
    test_ds = domain_data[test_domain]
    train_doms = [d for d in domain_data if d != test_domain]
    train_ds = concatenate_datasets([domain_data[d] for d in train_doms])
    train_val = train_ds.train_test_split(test_size=val_size, seed=SEED)
    return DatasetDict({
        "train": train_val["train"],
        "validation": train_val["test"],
        "test": test_ds,
    })

def prepare_lodo_experiment(domain_data, test_domain, tokenizer, label2id):
    splits = create_lodo_splits(domain_data, test_domain)
    tokenized = {}
    for split_name, ds in splits.items():
        ds = ds.map(
            lambda x: tokenize_and_align_labels(x, tokenizer, label2id),
            batched=True,
            remove_columns=ds.column_names,
        )
        tokenized[split_name] = ds
    return DatasetDict(tokenized)

**Метрики**

In [16]:
def compute_metrics(p):
    preds, labels = p
    preds = np.argmax(preds, axis=2)

    y_true, y_pred = [], []
    for pseq, lseq in zip(preds, labels):
        t, p = [], []
        for pi, li in zip(pseq, lseq):
            if li == -100:
                continue
            t.append(id2label[li])
            p.append(id2label[pi])
        y_true.append(t)
        y_pred.append(p)

    return {
        "f1": f1_score(y_true, y_pred),
        "precision": precision_score(y_true, y_pred),
        "recall": recall_score(y_true, y_pred),
    }

**Обучение для одного LODO-домена**

In [25]:
def run_lodo_experiment(domain_data, test_domain, tokenizer):
    model = AutoModelForTokenClassification.from_pretrained(
        MODEL_NAME,
        num_labels=len(label2id),
        id2label=id2label,
        label2id=label2id,
    )

    dataset = prepare_lodo_experiment(domain_data, test_domain, tokenizer, label2id)

    training_args = TrainingArguments(
        output_dir=f"./results_lodo_{test_domain}",
        eval_strategy="epoch",
        save_strategy="epoch",
        learning_rate=LEARNING_RATE,
        per_device_train_batch_size=BATCH_SIZE,
        per_device_eval_batch_size=BATCH_SIZE * 2,
        num_train_epochs=3,
        weight_decay=0.01,
        logging_steps=20,
        load_best_model_at_end=True,
        metric_for_best_model="f1",
        greater_is_better=True,
        report_to="none",
        seed=SEED,
        fp16=FP16,
    )

    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=dataset["train"],
        eval_dataset=dataset["validation"],
        compute_metrics=compute_metrics,
        callbacks=[EarlyStoppingCallback(early_stopping_patience=1)],
    )

    trainer.train()
    test_results = trainer.evaluate(dataset["test"])
    return test_results

**Полный кросс‑доменный цикл**

In [26]:
results = {}
for test_domain in DOMAIN_MAP.values():
    print(f"\n===== Тестовый домен: {test_domain} =====")
    metrics = run_lodo_experiment(datasets, test_domain, tokenizer)
    results[test_domain] = metrics
    print(f"F1={metrics['eval_f1']:.4f}, Precision={metrics['eval_precision']:.4f}, Recall={metrics['eval_recall']:.4f}")

avg_f1 = np.mean([r["eval_f1"] for r in results.values()])
avg_prec = np.mean([r["eval_precision"] for r in results.values()])
avg_rec = np.mean([r["eval_recall"] for r in results.values()])
print(f"\n===== Средние метрики =====")
print(f"F1={avg_f1:.4f}, Precision={avg_prec:.4f}, Recall={avg_rec:.4f}")


===== Тестовый домен: corruption =====


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

BertForTokenClassification LOAD REPORT from: ai-forever/ruBert-base
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.decoder.bias               | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
bert.pooler.dense.weight                   | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
bert.pooler.dense.bias                     | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loadi

Map:   0%|          | 0/10944 [00:00<?, ? examples/s]

Map:   0%|          | 0/1217 [00:00<?, ? examples/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.output.LayerNorm.weight', 'bert.encoder.layer.0.output.LayerNorm.bias', 'bert.encoder.layer.1.attention.output.LayerNorm.weight', 'bert.encoder.layer.1.attention.output.LayerNorm.bias', 'bert.encoder.layer.1.output.LayerNorm.weight', 'bert.encoder.layer.1.output.LayerNorm.bias', 'bert.encoder.layer.2.attention.output.LayerNorm.weight', 'bert.encoder.layer.2.attention.output.LayerNorm.bias', 'bert.encoder.layer.2.output.LayerNorm.weight', 'bert.encoder.layer.2.output.LayerNorm.bias', 'bert.encoder.layer.3.attention.output.LayerNorm.weight', 'bert.encoder.layer.3.attention.output.LayerNorm.bias', 'bert.encoder.layer.3.output.LayerNorm.weight', 'bert.encoder.layer.3.output.LayerNorm.bias', 'bert.encoder.layer.4.attention.output.La

F1=0.2968, Precision=0.3218, Recall=0.2754

===== Средние метрики =====
F1=0.3140, Precision=0.5122, Recall=0.2384
